In [1]:
# 📌 Email Spam Detector with Dataset Cleaning + Model Training

import pandas as pd
import string, re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Load dataset
data = pd.read_csv("spam.csv", encoding='latin-1')

# 2. Keep only useful columns
data = data[['v1', 'v2']]
data.columns = ['label', 'message']

# 3. Convert labels: ham -> 0, spam -> 1
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

# 4. Drop missing values
data.dropna(inplace=True)

# 5. Clean text function
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = text.strip()
    return text

data['message'] = data['message'].apply(clean_text)

# 6. Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(data['message'])
y = data['label']

# 7. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8. Train model (Naive Bayes)
model = MultinomialNB()
model.fit(X_train, y_train)

# 9. Evaluate model
y_pred = model.predict(X_test)

print("✅ Model Training Complete!\n")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 10. Function to predict new messages
def predict_email(text):
    text = clean_text(text)
    text_vec = vectorizer.transform([text])
    prediction = model.predict(text_vec)[0]
    return "Spam" if prediction == 1 else "Ham"

# 🔹 Example tests
print("\nTest Predictions:")
print(predict_email("Congratulations! You won a free iPhone. Click here to claim now!"))
print(predict_email("Hey, are we meeting tomorrow?"))


✅ Model Training Complete!

Accuracy: 0.968609865470852

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.77      0.87       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.93      1115
weighted avg       0.97      0.97      0.97      1115


Test Predictions:
Spam
Ham
